![Week 6 Day 3](../../assets/banners/w6d3.png)

# Week 6 Day 3 - Solution: A Resilient Progress Watcher

**The exercise:** follow a batch's SSE stream with one text progress bar per student, redrawn in place; then drop the connection after 10 events, reconnect with `Last-Event-ID`, and prove that no event was lost or seen twice.

Try the exercise yourself first: struggling a little is where the learning happens! Then compare with this one. Everything runs offline in 🧪 demo mode.

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ No {key_name} set. That's fine this week: every cell runs offline in 🧪 demo mode.")
        print("   (To grade for real later, see Part 5 of the setup guide, or setup/SWAP_PROVIDER.md)")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json
import shutil
import time

from fastapi.testclient import TestClient

from gradewise.api.main import create_app
from gradewise.config import OUTPUT_DIR
from gradewise.console import success

SANDBOX = OUTPUT_DIR / "week6" / "day3_solution"
shutil.rmtree(SANDBOX, ignore_errors=True)  # 💡 a fresh start every time you run the notebook

app = create_app(SANDBOX, demo=True)  # 🧪 demo=True: offline fakes, no API key, no cost
client = TestClient(app)  # 💡 calls the app in-process: real HTTP semantics, no server, no port


def show(data, limit: int = 1500) -> None:
    """Pretty-print JSON, cut short so a big response doesn't flood the notebook."""
    text = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n  …")


def wait_for(batch_id: str, *states: str, timeout: float = 120) -> dict:
    """Poll GET /batches/{id} until the batch reaches one of `states` (default: paused, done or failed)."""
    states = states or ("paused", "done", "failed")
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        batch = client.get(f"/batches/{batch_id}").json()
        if batch["status"] in states:
            return batch
        time.sleep(0.2)
    raise TimeoutError(f"{batch_id} never reached {states}")


success("A private GradeWise API for this notebook", {"Mode": "🧪 demo (offline fakes)", "Writes to": str(SANDBOX)})

In [ ]:


def read_sse(lines):
    """(event_name, data) pairs from SSE lines, like gradewise/ui/client.py."""
    name, data = "message", []
    for line in lines:
        if line.startswith(":"):
            continue
        if line.startswith("event:"):
            name = line[6:].strip()
        elif line.startswith("data:"):
            data.append(line[5:].strip())
        elif not line and data:
            yield name, json.loads("\n".join(data))
            name, data = "message", []


def text_bar(p: dict, width: int = 20) -> str:
    filled = round(width * min(100, p["percent"]) / 100)
    return f"{p['student_id']:<11} {'█' * filled}{'░' * (width - filled)} {p['percent']:>3}%  {p['stage']}"


class Watcher:
    """Follows one batch: keeps the last event ID, every ID seen, and the latest progress per student."""

    def __init__(self, batch_id: str):
        self.batch_id, self.last_id, self.seen, self.connections = batch_id, 0, [], 0
        self.progress = {p["student_id"]: p for p in client.get(f"/batches/{batch_id}").json()["progress"]}
        self.handle = display(Markdown(self.render()), display_id=True)

    def render(self) -> str:
        return "```text\n" + "\n".join(text_bar(p) for p in self.progress.values()) + "\n```"

    def follow(self, max_events: int | None = None) -> str | None:
        """Read the stream from where we left off. Returns the final status, or None if we 'dropped' early."""
        self.connections += 1
        headers = {"Last-Event-ID": str(self.last_id)} if self.last_id else {}
        with client.stream("GET", f"/batches/{self.batch_id}/events", headers=headers) as response:
            for count, (name, event) in enumerate(read_sse(response.iter_lines()), start=1):
                if name == "end":
                    return event["status"]
                self.last_id = event["id"]
                self.seen.append(event["id"])
                if event.get("progress"):
                    self.progress[event["student_id"]] = event["progress"]
                if event["stage"] == "done":
                    for p in self.progress.values():
                        p.update(stage="done", percent=100)
                self.handle.update(Markdown(self.render()))
                if max_events and count >= max_events:
                    return None  # 💥 leaving the `with` block closes the connection: our simulated network blip
        return None

### Watch, drop, reconnect

In [ ]:
n_students = min(3, len(datasets.synthetic_students()))  # 💡 3, or 2 with DATA_SIZE=small
batch_id = client.post(
    "/batches", data={"exam_id": "cs_midterm", "from_transcripts": "true", "students": str(n_students)}
).json()["batch_id"]
watcher = Watcher(batch_id)

status = watcher.follow(max_events=10)
print(f"💥 connection dropped after event {watcher.last_id} (status {status})")
status = watcher.follow()  # 🔁 reconnect: sends Last-Event-ID, gets only what came after
print(f"🔁 reconnected and followed to the end of this run: {status}, last event {watcher.last_id}")

while status == "paused":  # 🧑‍🏫 approve each review, then keep watching
    client.post(f"/batches/{batch_id}/review", json={"approved": True}).raise_for_status()
    wait_for(batch_id)
    status = watcher.follow()
print(f"🏁 batch {status}")

### The proof: nothing lost, nothing twice

In [ ]:
with client.stream("GET", f"/batches/{batch_id}/events") as response:
    replay = [e["id"] for name, e in read_sse(response.iter_lines()) if name == "progress"]

print(f"events seen by the watcher : {len(watcher.seen)} (unique: {len(set(watcher.seen))})")
print(f"events in a full replay    : {len(replay)}")
assert watcher.seen == replay == list(range(1, len(replay) + 1)), "lost or duplicated events!"
print(f"✅ every event exactly once, in order, across {watcher.connections} connections")

💡 This only works because GradeWise **stores** events before streaming them (`BatchJob.events`) and numbers them. If the endpoint streamed straight from the running graph, a dropped connection would lose whatever happened while you were away.

🚀 **In production** the events would live in a Redis Stream per batch instead of a Python list (`gradewise/api/jobs.py` explains), and a browser's `EventSource` sends `Last-Event-ID` for you on every automatic reconnect.